# Sweep: CLS Pooling

Run this notebook in **Colab 1** while running `07_sweep_mean.ipynb` and `07_sweep_token.ipynb` in separate Colab runtimes.

All results are saved to Google Drive so they can be merged later.


## 1. Setup


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Clone or pull the repository
!git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
%cd Negation-Origin-Tracing


/content/Negation-Origin-Tracing


In [3]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 53.8 MB/s eta 0:00:00


In [4]:
# Set shared output location on Google Drive
import os
os.environ['DRIVE_OUTPUT'] = '/content/drive/MyDrive/NOT_results'

# Create the directory
!mkdir -p /content/drive/MyDrive/NOT_results

print(f"Results will be saved to: {os.environ['DRIVE_OUTPUT']}")


Results will be saved to: /content/drive/MyDrive/NOT_results


In [5]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


GPU available: True
GPU: Tesla T4


## 2. Download Data


In [6]:
# Download data if needed
import os
if not os.path.exists('data/raw/train/sst.parquet'):
    print("Downloading data...")
    !python src/data/download.py
else:
    print("Data already exists!")


Data already exists!


## 3. Run CLS Pooling Sweep

This trains probes on all 6 layers using CLS pooling.


In [7]:
# Run the CLS sweep
!chmod +x run_sweep_cls.sh
!./run_sweep_cls.sh


Saving to Google Drive: /content/drive/MyDrive/NOT_results/sweep_cls
Layer Sweep: CLS Pooling
Layers: 0-5
Output: /content/drive/MyDrive/NOT_results/sweep_cls


[Layer 0] CLS pooling...
2025-12-02 21:35:53.031454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764711353.057839    1979 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764711353.063756    1979 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764711353.079353    1979 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764711353.079368    1979 computation_placer.cc:177] computation

## 4. Check Results


In [8]:
import json
import os

results_file = os.path.join(os.environ['DRIVE_OUTPUT'], 'sweep_cls', 'results_cls.json')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)

    print(f"CLS Sweep Results ({len(results)} experiments)")
    print("=" * 50)

    for r in sorted(results, key=lambda x: x.get('test_auroc', 0), reverse=True):
        print(f"Layer {r['layer']}: AUROC={r.get('test_auroc', 0):.4f}, Acc={r.get('test_acc', 0):.4f}")

    best = max(results, key=lambda x: x.get('test_auroc', 0))
    print(f"\nBest: Layer {best['layer']} with AUROC {best.get('test_auroc', 0):.4f}")
else:
    print(f"Results not found at {results_file}")


CLS Sweep Results (6 experiments)
Layer 0: AUROC=0.0000, Acc=0.0000
Layer 1: AUROC=0.0000, Acc=0.0000
Layer 2: AUROC=0.0000, Acc=0.0000
Layer 3: AUROC=0.0000, Acc=0.0000
Layer 4: AUROC=0.0000, Acc=0.0000
Layer 5: AUROC=0.0000, Acc=0.0000

Best: Layer 0 with AUROC 0.0000
